In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deeprank.torch import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=8, n_informative=6)

qt = QuantileTransformer()
y = qt.fit_transform(y.reshape(-1, 1))[:, 0]
y = np.floor(y * 4).astype(np.int64)  # 4 ordinal classes: 0, 1, 2, 3
y = np.clip(y, 0, 3)

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.25)

train_ds = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(x_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
print("Class distribution:", np.bincount(y_train))

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    OrdinalOutput(input_dim=16, output_dim=4),
)
ordinal_layer = model[2]
optimizer = torch.optim.Adam(model.parameters())
print(model)

In [ ]:
epochs = 50
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    epoch_loss = []
    for xb, yb in train_loader:
        h = model[1](model[0](xb))  # hidden features
        logits = ordinal_layer.linear(h)
        thresholds = ordinal_layer.interior_thresholds
        loss = ordinal_loss(logits, yb, thresholds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    train_losses.append(np.mean(epoch_loss))

    model.eval()
    v_loss = []
    with torch.no_grad():
        for xb, yb in val_loader:
            h = model[1](model[0](xb))
            logits = ordinal_layer.linear(h)
            v_loss.append(ordinal_loss(logits, yb, thresholds).item())
    val_losses.append(np.mean(v_loss))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    x_val_t = torch.tensor(x_val, dtype=torch.float32)
    probs = model(x_val_t)
    preds = probs.argmax(dim=1).numpy()
accuracy = np.mean(preds == y_val)
print(f"Validation accuracy: {accuracy:.4f}")

In [ ]:
plt.plot(train_losses, '--', label='train_loss')
plt.plot(val_losses, label='val_loss')
plt.title('Training and validation loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend()
plt.show()